# Travail de groupe ETL

In [29]:
# Imports des bibliothèques

import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

all_files = sorted(glob.glob("*.csv"))

df_dic = {}
for file in all_files:
    nom = (
        file.replace("datasets/Olist/", "")
        .replace("olist_", "")
        .replace("_dataset", "")
        .replace(".csv", "")
    )
    df_dic[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

## 1 - Table Orders

In [30]:
print(df_dic["orders"].describe().round(2))
print("\n")
print(resume(df_dic["orders"]))

                                order_id                       customer_id  \
count                              99441                             99441   
unique                             99441                             99441   
top     66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   
freq                                   1                                 1   

       order_status order_purchase_timestamp    order_approved_at  \
count         99441                    99441                99281   
unique            8                    98875                90733   
top       delivered      2018-08-02 12:05:26  2018-02-27 04:31:10   
freq          96478                        3                    9   

       order_delivered_carrier_date order_delivered_customer_date  \
count                         97658                         96476   
unique                        81018                         95664   
top             2018-05-09 15:48:00           2018-05-08

TypeError: '>=' not supported between instances of 'str' and 'float'

### Recherche des ecart moyens entres les colonnes

In [32]:
orders = df_dic["orders"]

ecart_approved = (
    (
        pd.to_datetime(orders["order_approved_at"])
        - pd.to_datetime(orders["order_purchase_timestamp"])
    )
    .dropna()
    .mean()
)

ecart_delivered_carrier = (
    (
        pd.to_datetime(orders["order_delivered_carrier_date"])
        - pd.to_datetime(orders["order_approved_at"])
    )
    .dropna()
    .mean()
)

ecart_delivered_customer_1 = (
    (
        pd.to_datetime(orders["order_delivered_customer_date"])
        - pd.to_datetime(orders["order_approved_at"])
    )
    .dropna()
    .mean()
)

ecart_delivered_customer_2 = (
    (
        pd.to_datetime(orders["order_delivered_customer_date"])
        - pd.to_datetime(orders["order_delivered_carrier_date"])
    )
    .dropna()
    .mean()
)


print(f"Écart entre l'approbation et l'achat : {ecart_approved}")
print(
    f"Écart entre l'approbation et la livraison par le transporteur : {ecart_delivered_carrier}"
)
print(
    f"Écart entre l'approbation et la livraison au client : {ecart_delivered_customer_1}"
)
print(
    f"Écart entre la livraison par le transporteur et la livraison au client : {ecart_delivered_customer_2}"
)
print(
    f"Écart entre la livraison par le transporteur et la livraison au client : {ecart_delivered_carrier + ecart_delivered_customer_2}"
)

Écart entre l'approbation et l'achat : 0 days 10:25:08.739486910
Écart entre l'approbation et la livraison par le transporteur : 2 days 19:19:15.298850927
Écart entre l'approbation et la livraison au client : 12 days 03:07:42.855694470
Écart entre la livraison par le transporteur et la livraison au client : 9 days 07:55:59.254148743
Écart entre la livraison par le transporteur et la livraison au client : 12 days 03:15:14.552999670


### Remplacement des données manquantes par la moyenne

In [33]:
#PRODUCT COLUMN CLEANING


products = df_dic["products"]
col = products["product_photos_qty"]
col.fillna(0)

cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]


products["nb_identiques"] = products.groupby(cols, dropna=False)[cols[0]].transform("size") - 1

resultat = products.sort_values("nb_identiques", ascending=False)
print(resultat[["product_id", "product_category_name"] + cols + ["nb_identiques"]])





                             product_id        product_category_name  \
2400   772453e71330bbe06de41e9baf8495ac           relogios_presentes   
2535   e48cff5b3591c9466cd84fcea1f1b3e4               consoles_games   
2346   07d56eb519d315a2d8fb2f98f095e6bf  fashion_bolsas_e_acessorios   
2552   7e48d9554094858ee2269c4ef809f50e           relogios_presentes   
2582   3264eb8097906cea4af2f4a56e2447ed           relogios_presentes   
...                                 ...                          ...   
32912  ca977202f2d5bc9cfdc368d305e460b8                esporte_lazer   
32914  07ae32b18ffbde7c6c928721e8b956fb                esporte_lazer   
32915  d790fcd60d4743d309fedf7e7bb4c902                 beleza_saude   
32916  624a104988a613eba3eae6cd9132d397              cama_mesa_banho   
5      41d3672d4792049fa1779bb35283ed13        instrumentos_musicais   

       product_weight_g  product_length_cm  product_height_cm  \
2400              200.0               16.0                2.0   
2535 

In [34]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

combinaisons = products.value_counts(subset=cols, dropna=False)

print(combinaisons.head(1))
print(combinaisons.head(10))

# les combinaisons présentes au moins 2 fois
doublons = combinaisons[combinaisons > 1].index

# les lignes de products qui portent une de ces combinaisons
lignes_identiques = products[products.set_index(cols).index.isin(doublons)]

print(len(lignes_identiques), "produits partagent leurs 4 valeurs avec au moins un autre")

# répartition par catégorie (dropna=False fait apparaître les sans-catégorie, NaN)
repartition = lignes_identiques["product_category_name"].value_counts(dropna=False)
print(repartition)

# combien sont sans catégorie
print(lignes_identiques["product_category_name"].isna().sum(), "sont sans catégorie")
for i in range (10):
    print ("-")

sans_categorie = products[products["product_category_name"].isna()]

combinaisons1 = sans_categorie.value_counts(subset=cols, dropna=False)

print(combinaisons1.head(1))
print(combinaisons1.head(10))
total_identiques = combinaisons1[combinaisons1 > 1].sum()
print(total_identiques, "produits sans catégorie partagent leurs 4 valeurs avec au moins un autre")

product_weight_g  product_length_cm  product_height_cm  product_width_cm
200.0             16.0               2.0                11.0                208
Name: count, dtype: int64
product_weight_g  product_length_cm  product_height_cm  product_width_cm
200.0             16.0               2.0                11.0                208
250.0             16.0               2.0                11.0                 71
100.0             19.0               3.0                11.0                 66
300.0             16.0               2.0                11.0                 65
100.0             16.0               5.0                16.0                 65
3000.0            69.0               11.0               11.0                 65
300.0             16.0               16.0               16.0                 57
                  17.0               4.0                12.0                 56
350.0             17.0               4.0                12.0                 53
200.0             16.0      

In [35]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

resultat = (products.groupby(cols, dropna=False)
            .agg(nb_lignes=("product_category_name", "size"),
                 categorie_majoritaire=("product_category_name",
                                         lambda s: s.value_counts(dropna=False).index[0]),
                 nb_categories=("product_category_name", "nunique"))
            .reset_index())

resultat = resultat[resultat["nb_lignes"] > 1].sort_values("nb_lignes", ascending=False)
resultat["nb_majoritaire"] = (products.groupby(cols, dropna=False)["product_category_name"]
                              .agg(lambda s: s.value_counts(dropna=False).iloc[0])
                              .values[resultat.index])

print(resultat.to_string(index=False))

 product_weight_g  product_length_cm  product_height_cm  product_width_cm  nb_lignes                          categorie_majoritaire  nb_categories  nb_majoritaire
            200.0               16.0                2.0              11.0        208                    fashion_bolsas_e_acessorios             18              69
            250.0               16.0                2.0              11.0         71                             relogios_presentes              9              42
            100.0               19.0                3.0              11.0         66                                      telefonia              5              53
            100.0               16.0                5.0              16.0         65                    fashion_bolsas_e_acessorios              6              40
           3000.0               69.0               11.0              11.0         65                               moveis_decoracao              2              64
            300.0     

In [47]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

# prix par produit (au cas où pas encore fait)
order_items = df_dic['order_items']
prix_produit = order_items.groupby("product_id")["price"].median()
if "price" not in products.columns:
    products = products.merge(prix_produit.rename("price"), on="product_id", how="left")

g = products.groupby(cols, dropna=False)
cat_dans_groupe = g["product_category_name"].transform(lambda s: s.notna().sum())
nb_identiques   = g["product_category_name"].transform("size") - 1
prix_frere      = g["price"].transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None)

masque = (products["product_category_name"].isna()
          & products["product_name_lenght"].isna()
          & products["product_description_lenght"].isna()
          & (cat_dans_groupe == 0))

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(nb_identiques=nb_identiques[masque], prix=prix_frere[masque])
            .sort_values("nb_identiques", ascending=False))

print(len(resultat), "produits sans catégorie / nom / description et sans frère catégorisé")

affichage = (resultat.rename(columns={
                "product_id": "produit",
                "product_weight_g": "poids (g)",
                "product_length_cm": "long. (cm)",
                "product_height_cm": "haut. (cm)",
                "product_width_cm": "larg. (cm)",
                "nb_identiques": "nb frères"})
             .style
             .hide(axis="index")
             .format({"prix": "R$ {:.2f}",
                      "poids (g)": "{:.0f}", "long. (cm)": "{:.0f}",
                      "haut. (cm)": "{:.0f}", "larg. (cm)": "{:.0f}"},
                     na_rep="—")
             .background_gradient(subset=["nb frères"], cmap="Reds"))

display(affichage)

392 produits sans catégorie / nom / description et sans frère catégorisé


produit,poids (g),long. (cm),haut. (cm),larg. (cm),nb frères,prix
52a5346c9dae9cd560b103875b94d123,200,22,14,14,4,R$ 78.90
7af3e2da474486a3519b0cba9dea8ad9,200,22,14,14,4,R$ 78.90
14086fd4907ab4135efce70c2cd3c013,200,22,14,14,4,R$ 78.90
ea8062a20ca055f459f3a99eccdec4b5,200,22,14,14,4,R$ 78.90
3abf2d4698bf245577543ea01d9c7f16,200,22,14,14,4,R$ 78.90
03706dca83513062fe74967a71b5fc78,400,25,8,11,2,R$ 34.99
7285e977b36a2243203636b240772c5e,400,25,8,11,2,R$ 34.99
fcb72602e3e6f59e5fdb5c3a4a3de410,8750,36,27,54,2,R$ 170.00
e20be15ef3888b958c8a76a2f0edfc25,400,25,8,11,2,R$ 34.99
b29fcbd778387a0218e2f058e57bce75,8750,36,27,54,2,R$ 170.00


In [46]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

# --- prix par produit (médiane des ventes) ---
order_items = df_dic['order_items']
prix_produit = order_items.groupby("product_id")["price"].median()
if "price" not in products.columns:
    products = products.merge(prix_produit.rename("price"), on="product_id", how="left")

# --- infos sur les frères (mêmes 4 dimensions) ---
g = products.groupby(cols, dropna=False)

cat_dans_groupe = g["product_category_name"].transform(lambda s: s.notna().sum())  # frères catégorisés
nb_categories   = g["product_category_name"].transform("nunique")                  # catégories différentes
nb_identiques   = g["product_category_name"].transform("size") - 1
categorie_majoritaire = g["product_category_name"].transform(
    lambda s: s.value_counts().index[0] if s.notna().any() else None)
nb_prix    = g["price"].transform("nunique")
prix_frere = g["price"].transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None)

# --- produits sans catégorie / nom / description, mais avec au moins un frère catégorisé ---
masque = (products["product_category_name"].isna()
          & products["product_name_lenght"].isna()
          & products["product_description_lenght"].isna()
          & (cat_dans_groupe > 0)
          & products[cols].notna().all(axis=1))   # écarte les produits sans dimensions

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(categorie_probable=categorie_majoritaire[masque],
                    nb_categories=nb_categories[masque],
                    nb_identiques=nb_identiques[masque],
                    nb_prix=nb_prix[masque],
                    price_match=(nb_prix[masque] == 1),
                    prix=prix_frere[masque])
            .sort_values(["nb_categories", "price_match", "nb_identiques"],
                         ascending=[True, False, False]))

print(len(resultat), "produits sans catégorie/nom/description avec frère catégorisé")
print((resultat["nb_categories"] == 1).sum(), "ont des frères tous d'accord sur une seule catégorie")
affichage = (resultat.rename(columns={
                "product_id": "produit",
                "product_weight_g": "poids (g)",
                "product_length_cm": "long. (cm)",
                "product_height_cm": "haut. (cm)",
                "product_width_cm": "larg. (cm)",
                "categorie_probable": "catégorie probable",
                "nb_categories": "nb catég.",
                "nb_identiques": "nb frères",
                "price_match": "prix idem"})
             .style
             .hide(axis="index")
             .format({"prix": "R$ {:.2f}",
                      "poids (g)": "{:.0f}", "long. (cm)": "{:.0f}",
                      "haut. (cm)": "{:.0f}", "larg. (cm)": "{:.0f}"})
             .background_gradient(subset=["nb frères"], cmap="Blues")
             .map(lambda v: "background-color:#d8f3dc" if v is True else "", subset=["prix idem"]))

display(affichage)

217 produits sans catégorie/nom/description avec frère catégorisé
65 ont des frères tous d'accord sur une seule catégorie


produit,poids (g),long. (cm),haut. (cm),larg. (cm),catégorie probable,nb catég.,nb frères,nb_prix,prix idem,prix
b0a0c5dd78e644373b199380612c350a,1800,30,20,70,cama_mesa_banho,1,2,1,True,R$ 89.90
9df852a6c839bc3202227666d70c7cae,9150,49,32,35,pet_shop,1,1,1,True,R$ 39.99
2b29170916b7bbc58fd9078f0f4e7234,300,50,4,40,informatica_acessorios,1,1,1,True,R$ 39.90
6ab9d49c399f239bc88ea28c7568213f,250,22,2,32,moveis_decoracao,1,1,1,True,R$ 12.99
57398ec023989dbb7618acae002f3370,1000,27,10,20,papelaria,1,1,1,True,R$ 59.90
373449943bcfafb2d3ff7423408a1c9d,350,20,5,11,bebes,1,1,1,True,R$ 16.90
b5d652cabeb01249ad1cf910eef3230b,2000,39,42,39,moveis_decoracao,1,1,1,True,R$ 99.00
3d2d7eac865e47ff3939ea01aa56fb96,300,16,13,13,relogios_presentes,1,8,8,False,R$ 34.99
62fde58e97724f8b7519f3789eefa33f,600,35,35,15,fashion_roupa_masculina,1,4,2,False,R$ 49.90
c8316575e7ce05609496b6d0b9bce6bd,600,30,3,20,moveis_decoracao,1,4,2,False,R$ 45.90


In [38]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

nb_cat_groupe = (products.groupby(cols, dropna=False)["product_category_name"]
                 .transform("nunique"))

# nombre d'autres lignes avec les mêmes 4 valeurs
nb_identiques = (products.groupby(cols, dropna=False)["product_category_name"]
                 .transform("size") - 1)

categorie_frere = (products.groupby(cols, dropna=False)["product_category_name"]
                   .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None))

masque = (products["product_category_name"].isna() & (nb_cat_groupe == 1))

print(masque.sum(), "produits sans catégorie dont tous les frères sont du même groupe")

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(categorie_probable=categorie_frere[masque],
                    nb_identiques=nb_identiques[masque])
            .sort_values("nb_identiques", ascending=False))

print(resultat.to_string(index=False))

66 produits sans catégorie dont tous les frères sont du même groupe
                      product_id  product_weight_g  product_length_cm  product_height_cm  product_width_cm          categorie_probable  nb_identiques
3d2d7eac865e47ff3939ea01aa56fb96             300.0               16.0               13.0              13.0          relogios_presentes              8
62fde58e97724f8b7519f3789eefa33f             600.0               35.0               35.0              15.0     fashion_roupa_masculina              4
24b1fcad996ffb1ead90e324f264b783             200.0               39.0                9.0              11.0      informatica_acessorios              4
a1804276d9941ac0733cfd409f5206eb             600.0               35.0               35.0              15.0     fashion_roupa_masculina              4
c8316575e7ce05609496b6d0b9bce6bd             600.0               30.0                3.0              20.0            moveis_decoracao              4
2ce375bc4110d978ae33e7d48d63f599

In [44]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

# --- 1. un prix par produit (médiane des ventes) ---
order_items = df_dic['order_items']
prix_produit = order_items.groupby("product_id")["price"].median()

if "price" not in products.columns:          # évite price_x / price_y si on relance
    products = products.merge(prix_produit.rename("price"), on="product_id", how="left")

# --- 2. infos sur les frères (lignes partageant les 4 mêmes dimensions) ---
g = products.groupby(cols, dropna=False)

nb_cat_groupe   = g["product_category_name"].transform("nunique")          # catégories différentes
nb_identiques   = g["product_category_name"].transform("size") - 1          # autres frères
categorie_frere = g["product_category_name"].transform(
                      lambda s: s.dropna().iloc[0] if s.notna().any() else None)
nb_prix         = g["price"].transform("nunique")
prix_frere      = g["price"].transform(
                      lambda s: s.dropna().iloc[0] if s.notna().any() else None)

# --- 3. produits sans catégorie, récupérables (frères unanimes + dimensions renseignées) ---
masque = (products["product_category_name"].isna()
          & (nb_cat_groupe == 1)
          & products[cols].notna().all(axis=1))

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(categorie_probable=categorie_frere[masque],
                    nb_identiques=nb_identiques[masque],
                    nb_prix=nb_prix[masque],
                    price_match=(nb_prix[masque] == 1),
                    prix=prix_frere[masque])
            .sort_values(["price_match", "nb_identiques"], ascending=[False, False]))

print(len(resultat), "produits récupérables")
print(resultat["price_match"].sum(), "confirmés aussi par le prix (price_match = True)")
print(resultat.to_string(index=False))

65 produits récupérables
7 confirmés aussi par le prix (price_match = True)
                      product_id  product_weight_g  product_length_cm  product_height_cm  product_width_cm          categorie_probable  nb_identiques  nb_prix  price_match    prix
b0a0c5dd78e644373b199380612c350a            1800.0               30.0               20.0              70.0             cama_mesa_banho              2        1         True   89.90
9df852a6c839bc3202227666d70c7cae            9150.0               49.0               32.0              35.0                    pet_shop              1        1         True   39.99
2b29170916b7bbc58fd9078f0f4e7234             300.0               50.0                4.0              40.0      informatica_acessorios              1        1         True   39.90
6ab9d49c399f239bc88ea28c7568213f             250.0               22.0                2.0              32.0            moveis_decoracao              1        1         True   12.99
57398ec023989dbb7618acae